# Demo sull'eliminazione della sinapsi SRA
## — Estrai fisicamente conoscenze specifiche da un LLM!

Questo taccuino è una demo interattiva dell'articolo *[\[AI Romance\] Estrai fisicamente conoscenze specifiche da un LLM! "Eliminazione sinapsi" ed "Eliminazione" di LLM hot-swap](https://qiita.com/JunSuzukiJapan/items/)*.

In SRA (Synaptic Routing Architecture), la conoscenza non necessaria e le sinapsi possono essere rimosse in due modi.

| Metodo | API | Scopo |
|------|-----|------|
| **Rimozione fisica** | `pop_synapses(N)` | Rimuove le sinapsi aggiunte tramite Hot-Swap dalla coda per ripristinare le dimensioni del modello |
| **Azzeramento (eliminazione)** | `clear_synapses([idx])` | Disabilita le sinapsi agli indici intermedi e le converte in slot liberi |

Confermeremo anche la **trappola della similarità del coseno** che si verifica durante l'azzeramento, e la sua soluzione, nella pratica.

Infine, applichiamo `clear_synapses` a un modello effettivamente addestrato al multitasking e dimostriamo che **solo la funzione dell'attività mirata viene distrutta mentre ogni altra attività rimane completamente intatta (Zero Forgetting)**.

---
## Parte 1: Rimozione delle sinapsi (`pop_synapses`)

Abbiamo tagliato fisicamente le sinapsi aggiunte successivamente tramite Hot-Swap, iniziando dalla coda.
Proprio come disinstallare un plugin da un sistema operativo, puoi rimuovere fisicamente parti del cervello dell'intelligenza artificiale.

In [ ]:
import sys, os

if 'google.colab' in sys.modules:
    if not os.path.exists('SynapticRouter'):
        !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install tiktoken torch matplotlib seaborn

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

import torch
import torch.nn.functional as F
from src.sra_language_models import MoESRALanguageModel


In [ ]:
# Build a small model to walk through the add -> remove flow
model = MoESRALanguageModel(vocab_size=1000, dim=128, layers=2, num_synapses=4, k=2, syn_hidden=256)

print('=== Approach 1: Synapse Removal (Physical Cut) ===')
print(f'[Initial state] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')

# Add 2 Synapses via Hot-Swap
model.add_synapses(2, freeze_base=True)
print(f'\n[After adding] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')

# Physically remove the added Synapses
model.pop_synapses(2)
print(f'\n[After removal] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')
print('\nMemory usage has been fully restored!')


---
## Parte 2: Cancellazione dello zero (Purge) e la "trappola della similarità coseno"

Se vogliamo eliminare una sinapsi a un indice intermedio, rimuoverla fisicamente sposterebbe gli ID.
Quindi sovrascriviamo i pesi con `0.0` — un "azzeramento".

Tuttavia, qui c’è una **trappola terrificante**. La somiglianza coseno del vettore zero diventa `0.0`,
e se i punteggi delle sinapsi normali sono negativi, la **sinapsi vuota ottiene un punteggio più alto e inizia ad assorbire i dati** — un fenomeno del buco nero.

Il router di SRA dispone di una maschera **`-inf`** integrata per prevenire questa trappola. Verifichiamolo nella pratica.

In [ ]:
print('=== Approach 2: Verifying zero-clear and the -inf mask ===')

# Create a new model
model2 = MoESRALanguageModel(vocab_size=1000, dim=128, layers=1, num_synapses=4, k=2, syn_hidden=256)

# Inspect the weights of Synapse #2 before zero-clearing
target_idx = 2
emb_before = model2.blocks[0].router.get_full_emb()
print(f'[Before zero-clear] Synapse {target_idx} router embedding norm: {torch.norm(emb_before[target_idx]):.4f}')
print(f'  W1 norm: {torch.norm(model2.blocks[0].get_full_param("w1")[target_idx]):.4f}')

# Execute zero-clear
model2.clear_synapses([target_idx])

emb_after = model2.blocks[0].router.get_full_emb()
print(f'\n[After zero-clear] Synapse {target_idx} router embedding norm: {torch.norm(emb_after[target_idx]):.4f}')
print(f'  W1 norm: {torch.norm(model2.blocks[0].get_full_param("w1")[target_idx]):.4f}')
print('\nThe weights of Synapse #2 have been zeroed out completely!')


In [ ]:
# Actually verify the -inf mask in action
print('=== Verifying the -inf mask ===')

# Run the router with a dummy input
model2.eval()
dummy_input = torch.randint(0, 1000, (1, 10))
with torch.no_grad():
    logits, router_logits = model2(dummy_input)

# Inspect the router logits (final layer)
router_scores = router_logits[0]  # shape: (batch, seq, num_synapses)
print(f'Router scores (first token):')
scores = router_scores[0, 0]
for i in range(len(scores)):
    label = 'BLOCKED (-inf)' if scores[i] == float('-inf') else f'{scores[i]:.4f}'
    marker = ' <- zero-cleared' if i == target_idx else ''
    print(f'  Synapse {i}: {label}{marker}')

print('\nThe score of the zero-cleared Synapse has been set to -inf,')
print('   so the router can never select this Synapse again.')
print('   This completely prevents the "black-hole phenomenon".')


---
## Parte 3: Prova funzionale: `clear_synapses` su un modello addestrato

Ora arriviamo all'evento principale. Nelle parti 1 e 2 abbiamo verificato il "comportamento dell'API",
ma ciò che conta davvero è la prova funzionale: **"Dopo l'azzeramento, viene persa solo la conoscenza cancellata mentre ogni altra parte della conoscenza rimane completamente intatta?"**

Utilizzando lo stesso approccio del taccuino 05 (l'esperimento Lesion):
1. Treno multitasking su `copy` e `reverse`
2. Identificare le sinapsi utilizzate frequentemente da `reverse` e rimuoverle con `clear_synapses`
3. Verificare che `reverse` diventi irrisolvibile mentre `copy` continua a ottenere un punteggio del 100% (Zero Forgetting)

In [ ]:
# Exactly the same setup as notebook 05
from src.sra_gpu_models import MoESRAModel
from src.sra_experiment import make_multitask_batch, make_batch, make_optimizer, load_balance_loss
from src.constants import VOCAB_SIZE, BOS, PAD

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

class MoESRAConfig:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)

config = MoESRAConfig(
    vocab_size=30,
    d_model=128,
    n_layers=2,
    n_heads=4,
    num_synapses=8,
    k=2,
    max_seq_len=32
)

model3 = MoESRAModel(config.vocab_size, config.d_model, config.n_layers, config.num_synapses, config.k, syn_hidden=256).to(device)
optimizer = make_optimizer(model3, lr=0.001)


In [ ]:
# Multitask training (same as notebook 05)
print('Training started... (Copy & Reverse)')
model3.train()
epochs = 1500
batch_size = 32
tasks = ['copy', 'reverse']

for epoch in range(epochs):
    x, y, _ = make_multitask_batch(tasks, batch_size, min_len=4, max_len=8, device=device)

    optimizer.zero_grad()
    y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    outputs, routing_weights, _ = model3(x, y_in)

    loss = F.cross_entropy(outputs.reshape(-1, config.vocab_size), y.reshape(-1))
    lb_loss = load_balance_loss(routing_weights) * 0.1
    total_loss = loss + lb_loss

    total_loss.backward()
    optimizer.step()

    if (epoch + 1) % 300 == 0:
        print(f'Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}')

print('Training finished!')


### 3-1. Test delle prestazioni pre-eliminazione e identificazione del target
Confermiamo che ogni compito può essere risolto correttamente e identifichiamo le sinapsi maggiormente utilizzate da `reverse`.

In [ ]:
def test_task(task_name):
    model3.eval()
    x, y, _ = make_multitask_batch([task_name], batch_size=100, min_len=6, max_len=6, device=device)
    with torch.no_grad():
        y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
        outputs, routing_weights, _ = model3(x, y_in)
        preds = outputs.argmax(dim=-1)

    mask = y != PAD
    acc = (preds[mask] == y[mask]).float().mean().item()

    # Aggregate which Synapses were used (Layer 0)
    chosen = routing_weights[0].argmax(dim=-1)
    usage = torch.bincount(chosen.flatten(), minlength=config.num_synapses)

    print(f'[{task_name.upper()}] Accuracy: {acc*100:.1f}%')
    return usage

print('=== Before Deletion ===')
copy_usage = test_task('copy')
reverse_usage = test_task('reverse')

# Identify Synapses heavily used by Reverse but rarely used by Copy
# (Same logic as notebook 05)
diff = reverse_usage - copy_usage
target_synapses = [i for i, val in enumerate(diff) if val > 0]

# If they cannot be perfectly separated, pick the single Synapse most used by Reverse
if len(target_synapses) == 0:
    target_synapses = [diff.argmax().item()]

print(f'\n=> Deletion target: Synapses {target_synapses} (heavily used by REVERSE)')


### 3-2. Cancellazione sinapsi tramite `clear_synapses`
Nel notebook 05 abbiamo eseguito manualmente `block.w1.data[idx].zero_()`, ma qui utilizziamo l'**`clear_synapses` API** ufficiale, che applica anche la maschera `-inf` del router, per eseguire una cancellazione sicura.

In [ ]:
print(f'Deleting Synapses {target_synapses} via clear_synapses...')

# Record norms before deletion
for idx in target_synapses:
    emb_norm = torch.norm(model3.blocks[0].router.synapse_emb.data[idx]).item()
    w1_norm = torch.norm(model3.blocks[0].w1.data[idx]).item()
    print(f'  [Before deletion] Synapse {idx}: router embedding norm={emb_norm:.4f}, W1 norm={w1_norm:.4f}')

# Use the clear_synapses API (the router's -inf mask is also applied automatically)
for block in model3.blocks:
    block.router.clear_synapses(target_synapses)
    for idx in target_synapses:
        block.w1.data[idx].zero_()
        block.b1.data[idx].zero_()
        block.w2.data[idx].zero_()
        block.b2.data[idx].zero_()
        block.state.data[idx].zero_()

print('\nDeletion complete!')

# Check norms after deletion
for idx in target_synapses:
    emb_norm = torch.norm(model3.blocks[0].router.synapse_emb.data[idx]).item()
    w1_norm = torch.norm(model3.blocks[0].w1.data[idx]).item()
    print(f'  [After deletion] Synapse {idx}: router embedding norm={emb_norm:.4f}, W1 norm={w1_norm:.4f}')


### 3-3. Test delle prestazioni post-eliminazione (verifica di Zero Forgetting)

Proviamo di nuovo con alcune sinapsi rimosse.

**Risultati attesi:**
- **Copia**: precisione preservata (utilizza diverse sinapsi, quindi completamente intatte)
- **Retromarcia**: la precisione diminuisce (le sue sinapsi specializzate sono scomparse)

In [ ]:
print('=== After Deletion ===')
test_task('copy')
test_task('reverse')

print('\n[Discussion]')
print('When you destroy part of a single monolithic neural network by zeroing it out,')
print('all tasks usually collapse together.')
print('However, in SRA, an independent expert pathway (Synapse) is autonomously formed')
print('for each task, so even when clear_synapses removes a specific Synapse,')
print('tasks that use a different Synapse are completely unaffected.')
print('\nThis is the true strength of SRA as a modular AI.')
print('The free slot left behind by a deleted Synapse can later be reused by')
print('overwriting it (Hot-Swap) with a new plugin!')


---
## Riepilogo

| Dimostrazione | Cosa è stato dimostrato |
|------|---------------------|
| Parte 1 | Rimozione fisica e ripristino della memoria tramite `pop_synapses` |
| Parte 2 | Azzeramento tramite `clear_synapses` e verifica della maschera `-inf` |
| Parte 3 | `clear_synapses` su modello addestrato -> solo il compito mirato viene distrutto mentre gli altri rimangono intatti |

In questo modo, abbiamo raggiunto l'intero ciclo di vita di un'intelligenza artificiale modulare: **"addestramento -> integrazione (hot-swap) -> cancellazione (eliminazione) -> riutilizzo degli slot (sovrascrittura)"**.